# AO3 Tag Analysis

Notebook version of `ao3_tag_analysis.py`. Reads an existing `ao3_tag_metadata.csv`
(produced by `ao3_tag_scraper.py`) and runs two further analyses beyond
`ao3_tag_visualizer.py`'s bipartite network and tag-pair lift/PMI network:

1. **additional_tags frequency ranking** -- which values are most common, and
   which are least common (excluding one-off singletons)
2. **Cross-field hierarchical clustering** -- pools labels from *all* metadata
   fields (`rating`, `warnings`, `category`, `fandom`, `relationship`,
   `character`, `additional_tags`) and clusters them by lift/PMI similarity,
   rendered as a dendrogram + reordered heatmap plus a discrete
   cluster-membership CSV.

No network access is required -- this only reads a local CSV. Functions
reused from `ao3_tag_visualizer.py`'s tag-pair-statistics machinery are
copied inline below (matching this repo's existing notebook convention),
each marked with a comment noting the source -- keep them in sync if that
file changes.

In [ ]:
# Installs any of this notebook's dependencies that aren't already present
# (pandas, numpy, scipy, networkx, pyvis). Safe to re-run.
%pip install -q pandas numpy scipy networkx pyvis

In [ ]:
import heapq
import json
import math
import os
import re
import sys

import networkx as nx
import numpy as np
import pandas as pd
import scipy.sparse as sp
from networkx.algorithms.community import louvain_communities
from pyvis.network import Network
from pyvis.node import Node
from pyvis.edge import Edge

## Configuration

Edit these values, then run all cells in order.

In [ ]:
INPUT = "ao3_tag_metadata.csv"

FREQUENCY_TOP_N = 20          # most frequent additional_tags to report, per
                              # category -- seed tags and non-seed tags each
                              # get up to this many
FREQUENCY_BOTTOM_N = 20       # least frequent additional_tags to report
FREQUENCY_MIN_COUNT = 2       # floor for "least frequent" -- excludes values with
                              # fewer works than this (default 2 excludes one-off
                              # singletons)
FREQUENCY_OUT = "ao3_additional_tags_frequency.csv"

TOP_TAGS = 60                 # top N tags overall, pooled across all 7 metadata
                              # fields, by document frequency, before clustering.
                              # Overridden by ALL_TAGS
ALL_TAGS = False              # cluster using every tag from all 7 metadata fields,
                              # ignoring TOP_TAGS
MIN_PAIR_COUNT = 2            # drop pairs co-occurring fewer than this many times
                              # before clustering
CLUSTER_RESOLUTION = 1.0      # Louvain resolution -- higher means more, smaller
                              # communities; lower means fewer, larger ones
MIN_CLUSTER_SIZE = 1          # merge communities smaller than this into another
                              # community (their strongest-connected neighbor, or the
                              # largest remaining community if fully isolated)
                              # (1 = no minimum)
CLUSTER_NETWORK_OUT = "ao3_tag_cluster_network.html"
CLUSTER_META_NETWORK_OUT = "ao3_tag_cluster_meta_network.html"
GEXF_OUT = None               # set to e.g. "ao3_tag_network.gexf" to also export
                              # the full tag graph for Gephi -- off by default,
                              # the XML is large/slow to write at ALL_TAGS scale
CLUSTERS_OUT = "ao3_tag_clusters.csv"

DELIMITER = ", "
ALL_METADATA_FIELDS = ["rating", "warnings", "category", "fandom",
                        "relationship", "character", "additional_tags"]
FIELD_COLORS = {
    "seed_tag": "#4C72B0", "rating": "#DD8452", "warnings": "#55A868",
    "category": "#C44E52", "fandom": "#8172B2", "additional_tags": "#937860",
    "character": "#CCB974", "relationship": "#64B5CD",
}
_CLUSTER_PALETTE = [
    "#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2",
    "#937860", "#CCB974", "#64B5CD", "#8C8C8C", "#E377C2",
]
_CLUSTER_SEED = 0

## Load metadata

The CSV is loaded once. Multi-value fields (e.g. `additional_tags`, `character`) are comma-delimited within a cell -- `explode_field` splits them into one row per (work, value) pair on demand.

In [ ]:
# copied from ao3_tag_visualizer.py -- keep in sync if that file changes
def load_metadata(input_csv):
    df = pd.read_csv(input_csv, dtype=str, keep_default_na=False)
    return df


def split_values(cell, delimiter=DELIMITER):
    if not cell:
        return []
    return [v.strip() for v in cell.split(delimiter) if v.strip()]


def explode_field(df, field):
    exploded = df[["work_id", field]].copy()
    exploded[field] = exploded[field].apply(split_values)
    exploded = exploded.explode(field)
    exploded = exploded[exploded[field].notna() & (exploded[field] != "")]
    return exploded


df = load_metadata(INPUT)
df.head()

## additional_tags frequency ranking

A flat frequency count of every `additional_tags` value across the whole dataset -- the most common values, and the least common ones (excluding one-off singletons per `FREQUENCY_MIN_COUNT`).

In [ ]:
# copied from ao3_tag_analysis.py -- keep in sync if that file changes
def additional_tags_frequency(df, min_bottom_count=2):
    # Seed tags are the values in df["tag"] (the AO3 tag actually searched
    # to find each work). An additional_tags value that also happens to be
    # a seed tag partly reflects the scrape's own search bias rather than a
    # genuinely emergent discovery, so it's split out from non-seed values.
    seed_tags = set(df["tag"].unique())
    exploded = explode_field(df, "additional_tags")
    counts = exploded["additional_tags"].value_counts()
    counts = counts.reset_index()
    counts.columns = ["additional_tags", "count"]

    def sort_desc(c):
        return c.sort_values(["count", "additional_tags"], ascending=[False, True])

    is_seed = counts["additional_tags"].isin(seed_tags)
    most_frequent_seed = sort_desc(counts[is_seed])
    most_frequent_non_seed = sort_desc(counts[~is_seed])

    least_frequent = counts[counts["count"] >= min_bottom_count]
    least_frequent = least_frequent.sort_values(["count", "additional_tags"],
                                                 ascending=[True, True])
    return most_frequent_seed, most_frequent_non_seed, least_frequent


def write_frequency_csv(most_frequent_seed, most_frequent_non_seed, least_frequent,
                         top_n, bottom_n, out_path):
    most_seed = most_frequent_seed.head(top_n).copy()
    most_seed["rank_type"] = "most_frequent_seed_tag"
    most_non_seed = most_frequent_non_seed.head(top_n).copy()
    most_non_seed["rank_type"] = "most_frequent_non_seed_tag"
    least = least_frequent.head(bottom_n).copy()
    least["rank_type"] = "least_frequent"
    combined = pd.concat([most_seed, most_non_seed, least], ignore_index=True)
    combined = combined[["rank_type", "additional_tags", "count"]]
    combined.to_csv(out_path, index=False)
    return combined


most_frequent_seed, most_frequent_non_seed, least_frequent = additional_tags_frequency(
    df, min_bottom_count=FREQUENCY_MIN_COUNT)
frequency_table = write_frequency_csv(most_frequent_seed, most_frequent_non_seed, least_frequent,
                                       FREQUENCY_TOP_N, FREQUENCY_BOTTOM_N, FREQUENCY_OUT)
print(f"wrote {FREQUENCY_OUT} "
      f"({min(FREQUENCY_TOP_N, len(most_frequent_seed))} most frequent seed tags, "
      f"{min(FREQUENCY_TOP_N, len(most_frequent_non_seed))} most frequent non-seed tags, "
      f"{min(FREQUENCY_BOTTOM_N, len(least_frequent))} least frequent)")
frequency_table

## Cross-field community detection (lift/PMI)

A different question from frequency ranking: not "how common is this value", but
"which labels -- of *any* metadata field, `rating`/`warnings`/`category` included --
statistically tend to appear together". Pools all seven fields into one namespaced
label space (`f"{field}::{value}"`, same scheme as `ao3_tag_visualizer.py`'s
`--tag-pairs`), computes pairwise lift/PMI:

- `lift(A, B) = P(A, B) / (P(A) * P(B))`
- `pmi(A, B) = log2(lift(A, B))`

...then groups labels by similarity via graph-based community detection
(networkx's Louvain, operating directly on the sparse co-occurrence graph -- no
dense tag x tag matrix is ever built, unlike hierarchical clustering, which
needs one and can't scale past a few thousand tags). `apply_min_pair_count`
still runs first (dropping low-sample coincidences whose lift is statistically
meaningless); an edge is added only for `pmi > 0` pairs -- a community of
co-occurring tags should only be built from affinity edges.

In [ ]:
# copied from ao3_tag_visualizer.py -- keep in sync if that file changes
def build_document_tag_table(df, fields=ALL_METADATA_FIELDS):
    deduped = df.drop_duplicates(subset="work_id", keep="first")
    tables = []
    for field in fields:
        exploded = explode_field(deduped, field)
        exploded = exploded.rename(columns={field: "value"})
        exploded["tag_id"] = field + "::" + exploded["value"]
        tables.append(exploded[["work_id", "tag_id"]])
    return pd.concat(tables, ignore_index=True)


def top_k_tags_by_document_frequency(tag_table, k):
    if k is None:
        return None
    counts = tag_table["tag_id"].value_counts().reset_index()
    counts.columns = ["tag_id", "count"]
    counts = counts.sort_values(["count", "tag_id"], ascending=[False, True])
    return set(counts["tag_id"].head(k))


def build_tag_incidence_matrix(tag_table, keep_tags):
    filtered = tag_table[tag_table["tag_id"].isin(keep_tags)]
    columns = sorted(keep_tags)
    work_ids = sorted(filtered["work_id"].unique())
    col_index = {tag_id: i for i, tag_id in enumerate(columns)}
    row_index = {work_id: i for i, work_id in enumerate(work_ids)}

    row_idx = filtered["work_id"].map(row_index).to_numpy()
    col_idx = filtered["tag_id"].map(col_index).to_numpy()
    matrix = sp.coo_matrix(
        (np.ones(len(filtered), dtype=np.int8), (row_idx, col_idx)),
        shape=(len(work_ids), len(columns)), dtype=np.int8,
    ).tocsr()
    # Collapse any duplicate (work_id, tag_id) rows to a plain 0/1 presence
    # flag -- matches pd.crosstab(...) > 0's semantics regardless of raw
    # counts (coo_matrix sums duplicate entries on conversion to csr).
    matrix.data[:] = 1
    return pd.DataFrame.sparse.from_spmatrix(matrix, index=work_ids, columns=columns)


def tag_pair_statistics(incidence, n_docs):
    tags = list(incidence.columns)
    values = incidence.sparse.to_coo().tocsr().astype(np.int64)
    joint = values.T @ values
    marginal = np.asarray(joint.diagonal()).ravel()

    # scipy.sparse.triu only stores the upper triangle's nonzero entries --
    # equivalent to np.triu_indices(k=1) on the dense matrix, but without
    # ever materializing the full tags x tags array to get there (a dense
    # tags x tags int64 array is 48.9 GiB at 81,037 tags -- exactly the
    # MemoryError this sparse rewrite fixes).
    upper = sp.triu(joint, k=1).tocoo()
    i_idx, j_idx, joint_counts = upper.row, upper.col, upper.data
    nonzero = joint_counts > 0
    i_idx, j_idx, joint_counts = i_idx[nonzero], j_idx[nonzero], joint_counts[nonzero]

    count_a = marginal[i_idx]
    count_b = marginal[j_idx]
    lift = (joint_counts * n_docs) / (count_a * count_b)
    pmi = np.log2(lift)

    # Explicit canonicalization: tag_a is always the alphabetically-smaller
    # of the pair, regardless of the incidence matrix's own column order --
    # np.triu_indices above only guarantees i < j by column *position*, not
    # by alphabetical value. This used to be true only as an incidental side
    # effect of build_tag_incidence_matrix always pre-sorting its columns;
    # enforcing it here means the guarantee holds for any caller, not just
    # today's one. lift/pmi are symmetric in count_a/count_b already, so
    # only the names and their paired counts need to swap together.
    tags_arr = np.array(tags)
    a_names = tags_arr[i_idx]
    b_names = tags_arr[j_idx]
    swap = a_names > b_names
    tag_a = np.where(swap, b_names, a_names)
    tag_b = np.where(swap, a_names, b_names)
    count_a_final = np.where(swap, count_b, count_a)
    count_b_final = np.where(swap, count_a, count_b)

    return pd.DataFrame({
        "tag_a": tag_a,
        "tag_b": tag_b,
        "joint_count": joint_counts,
        "count_a": count_a_final,
        "count_b": count_b_final,
        "lift": lift,
        "pmi": pmi,
    })


def apply_min_pair_count(pair_stats, min_pair_count):
    return pair_stats[pair_stats["joint_count"] >= min_pair_count]


# copied from ao3_tag_analysis.py -- keep in sync if that file changes
def build_all_fields_pair_data(df, top_tags, min_pair_count):
    tag_table = build_document_tag_table(df, fields=ALL_METADATA_FIELDS)
    keep_tags = top_k_tags_by_document_frequency(tag_table, top_tags)
    if keep_tags is None:
        keep_tags = set(tag_table["tag_id"].unique())
    incidence = build_tag_incidence_matrix(tag_table, keep_tags)
    n_docs = df["work_id"].nunique()
    pair_stats = tag_pair_statistics(incidence, n_docs)
    pair_stats = apply_min_pair_count(pair_stats, min_pair_count)
    return pair_stats, keep_tags


top_tags_value = None if ALL_TAGS else TOP_TAGS
pair_stats, keep_tags = build_all_fields_pair_data(df, top_tags_value, MIN_PAIR_COUNT)
print(f"{len(keep_tags)} tags kept, {len(pair_stats)} surviving pairs")

## Cluster network + cluster membership CSV

Every tag in `keep_tags` becomes a node upfront (so an isolated tag with no
surviving pairs still appears, e.g. under `ALL_TAGS`), colored/grouped by its
final `cluster_id` rather than its metadata field. The interactive network and
the CSV are built from the same `communities`, so they can't disagree with each
other. Reuses the same pyvis rendering + filter-panel machinery as
`ao3_tag_visualizer.py`'s `--tag-pairs` network (checkboxes filter by cluster
instead of by field).

In [ ]:
# copied from ao3_tag_visualizer.py -- keep in sync if that file changes
_BOOTSTRAP_TAG_RE = re.compile(
    r'<(?:link|script)[^>]*\bhttps://cdn\.jsdelivr\.net/npm/bootstrap[^>]*>(?:</script>)?\s*'
)


def _strip_bootstrap_cdn(html_path):
    # pyvis's bundled template.html unconditionally references Bootstrap from
    # a CDN for the "card"/"card-body" wrapper div, regardless of
    # cdn_resources (which only inlines vis-network's own JS/CSS). That div
    # is purely cosmetic -- the graph itself is vis-network, already inlined,
    # and has no Bootstrap dependency -- so strip the two tags to make the
    # file genuinely self-contained.
    with open(html_path, encoding="utf-8") as f:
        html = f.read()
    html = _BOOTSTRAP_TAG_RE.sub("", html)
    with open(html_path, "w", encoding="utf-8") as f:
        f.write(html)


_FILTER_PANEL_CSS = """
<style>
#ao3-filter-panel { font-family: sans-serif; font-size: 14px; padding: 10px 14px;
  border-bottom: 1px solid #ccc; background: #fafafa; }
/* Bounded + scrollable: harmless for a handful of checkboxes (no scrollbar
   when content fits), essential for the cluster network, where real
   --all-tags data can produce thousands of clusters -- an unbounded
   checkbox list fills pages and pushes the graph canvas below the fold. */
#ao3-cat-checkboxes { max-height: 108px; overflow-y: auto; }
#ao3-filter-panel .ao3-cat-label { margin-right: 14px; white-space: nowrap;
  display: inline-block; }
#ao3-filter-panel .ao3-swatch { display: inline-block; width: 10px; height: 10px;
  border-radius: 50%; margin: 0 4px 0 4px; vertical-align: middle; }
#ao3-tag-picker { margin-top: 8px; position: relative; }
#ao3-tag-picker .ao3-hint { color: #777; font-weight: normal; font-size: 12px; }
#ao3-tag-chips { display: flex; flex-wrap: wrap; gap: 4px; margin: 6px 0; }
.ao3-tag-chip { background: #4C72B0; color: #fff; border-radius: 12px;
  padding: 2px 6px 2px 10px; font-size: 12px; display: inline-flex; align-items: center; }
.ao3-chip-remove { background: none; border: none; color: #fff; cursor: pointer;
  font-size: 14px; margin-left: 4px; line-height: 1; }
#ao3-tag-search { width: 260px; padding: 4px 6px; }
#ao3-tag-dropdown { position: absolute; z-index: 10; background: #fff; border: 1px solid #ccc;
  max-height: 220px; overflow-y: auto; width: 260px; }
.ao3-tag-option { padding: 4px 8px; cursor: pointer; }
.ao3-tag-option:hover { background: #eef; }
</style>
"""

_TAG_PAIR_FILTER_PANEL_HTML = """
<div id="ao3-filter-panel">
  <div id="ao3-cat-checkboxes">__CHECKBOX_ITEMS__</div>
  <div id="ao3-tag-picker">
    <strong>Tags</strong> <span class="ao3-hint">(none selected = show all;
    selecting narrows to those tags + their direct connections)</span>
    <div id="ao3-tag-chips"></div>
    <input type="text" id="ao3-tag-search" placeholder="Search tags…" autocomplete="off">
    <div id="ao3-tag-dropdown" hidden></div>
  </div>
</div>
""" + _FILTER_PANEL_CSS

_TAG_PAIR_FILTER_SCRIPT_TEMPLATE = """
<script>
(function () {
  const ALL_TAGS = __ALL_TAGS_JSON__;
  let selectedTagIds = [];

  function buildNodeGroups() {
    const map = {};
    for (const n of nodes.get()) map[n.id] = n.group;
    return map;
  }

  function buildFullAdjacency() {
    // Every node here is a "tag" -- unlike the bipartite graph, there's no
    // privileged always-visible node class, so adjacency is symmetric and
    // built for every node, not just one side of a bipartite split.
    const adj = {};
    for (const e of edges.get()) {
      (adj[e.from] = adj[e.from] || new Set()).add(e.to);
      (adj[e.to] = adj[e.to] || new Set()).add(e.from);
    }
    return adj;
  }

  const NODE_GROUPS = buildNodeGroups();
  const FULL_ADJACENCY = buildFullAdjacency();

  function applyFilters() {
    const checkedGroups = new Set(
      Array.from(document.querySelectorAll('.ao3-cat-checkbox:checked'))
           .map(function (cb) { return cb.dataset.group; })
    );
    let reachable = null;
    if (selectedTagIds.length) {
      reachable = new Set();
      selectedTagIds.forEach(function (id) {
        reachable.add(id);
        const neighbors = FULL_ADJACENCY[id];
        if (neighbors) neighbors.forEach(function (n) { reachable.add(n); });
      });
    }
    const updates = [];
    for (const nodeId in NODE_GROUPS) {
      let hidden = !checkedGroups.has(NODE_GROUPS[nodeId]);
      if (!hidden && reachable) hidden = !reachable.has(nodeId);
      updates.push({ id: nodeId, hidden: hidden });
    }
    nodes.update(updates); // vis-network auto-hides edges with a hidden endpoint
  }

  function tagById(id) {
    return ALL_TAGS.find(function (t) { return t.id === id; });
  }

  function matchingTags(query) {
    const q = query.trim().toLowerCase();
    const available = ALL_TAGS.filter(function (t) { return selectedTagIds.indexOf(t.id) === -1; });
    if (!q) return available;
    return available.filter(function (t) {
      return t.label.toLowerCase().indexOf(q) !== -1 || t.field.toLowerCase().indexOf(q) !== -1;
    });
  }

  function renderDropdown(query) {
    const dropdown = document.getElementById('ao3-tag-dropdown');
    dropdown.innerHTML = "";
    const matches = matchingTags(query);
    matches.forEach(function (t) {
      const opt = document.createElement('div');
      opt.className = 'ao3-tag-option';
      opt.textContent = t.label + " (" + t.field + ")";
      opt.dataset.id = t.id;
      dropdown.appendChild(opt);
    });
    dropdown.hidden = matches.length === 0;
  }

  function renderChips() {
    const chips = document.getElementById('ao3-tag-chips');
    chips.innerHTML = "";
    selectedTagIds.forEach(function (id) {
      const t = tagById(id);
      const chip = document.createElement('span');
      chip.className = 'ao3-tag-chip';
      const label = document.createElement('span');
      label.textContent = t ? (t.label + " (" + t.field + ")") : id;
      const remove = document.createElement('button');
      remove.className = 'ao3-chip-remove';
      remove.textContent = '×';
      remove.dataset.id = id;
      chip.appendChild(label);
      chip.appendChild(remove);
      chips.appendChild(chip);
    });
  }

  function addTag(id) {
    if (selectedTagIds.indexOf(id) === -1) selectedTagIds.push(id);
    renderChips();
    applyFilters();
  }

  function removeTag(id) {
    selectedTagIds = selectedTagIds.filter(function (x) { return x !== id; });
    renderChips();
    applyFilters();
  }

  const searchInput = document.getElementById('ao3-tag-search');
  searchInput.addEventListener('input', function () { renderDropdown(searchInput.value); });
  searchInput.addEventListener('focus', function () { renderDropdown(searchInput.value); });
  searchInput.addEventListener('click', function () { renderDropdown(searchInput.value); });

  document.getElementById('ao3-tag-dropdown').addEventListener('click', function (e) {
    const opt = e.target.closest('.ao3-tag-option');
    if (!opt) return;
    addTag(opt.dataset.id);
    searchInput.value = "";
    document.getElementById('ao3-tag-dropdown').hidden = true;
  });

  document.getElementById('ao3-tag-chips').addEventListener('click', function (e) {
    const btn = e.target.closest('.ao3-chip-remove');
    if (btn) removeTag(btn.dataset.id);
  });

  document.addEventListener('click', function (e) {
    const dropdown = document.getElementById('ao3-tag-dropdown');
    if (!dropdown.contains(e.target) && e.target !== searchInput) dropdown.hidden = true;
  });

  document.querySelectorAll('.ao3-cat-checkbox').forEach(function (cb) {
    cb.addEventListener('change', applyFilters);
  });

  applyFilters();
})();
</script>
"""


_STABILIZE_THEN_STOP_SCRIPT = """
<script>
(function () {
  network.once("stabilizationIterationsDone", function () {
    network.setOptions({ physics: false });
  });
})();
</script>
"""


def _inject_stabilize_then_stop(html_path):
    with open(html_path, encoding="utf-8") as f:
        html = f.read()
    assert "</body>" in html, "expected a </body> tag"
    html = html.replace("</body>", _STABILIZE_THEN_STOP_SCRIPT + "</body>", 1)
    with open(html_path, "w", encoding="utf-8") as f:
        f.write(html)


_NETWORK_OPTIONS_JSON = json.dumps({
    "layout": {"improvedLayout": False},
    "physics": {
        "solver": "barnesHut",
        "barnesHut": {"avoidOverlap": 0.5},
        "stabilization": {"enabled": True, "iterations": 200, "fit": True},
    },
})

# Used when the caller has already baked fixed x/y positions into every
# node -- physics off means vis-network places nodes at exactly those
# coordinates and never runs a client-side force simulation over them.
_STATIC_NETWORK_OPTIONS_JSON = json.dumps({
    "layout": {"improvedLayout": False},
    "physics": {"enabled": False},
})


def _fast_populate_network(net, graph, default_node_size=10, default_edge_weight=1):
    # Equivalent to pyvis's Network.from_nx(graph) -- confirmed to produce
    # byte-identical net.nodes/net.edges/net.node_ids/net.node_map for the
    # same input graph -- but without pyvis's own from_nx()/add_node()/
    # add_edge(), which check membership via `x in self.node_ids` and
    # `x in self.edges` (plain Python lists, not sets), an O(V) or O(E)
    # scan on every single node/edge insertion. At real --all-tags scale
    # that's catastrophic: confirmed directly, from_nx() alone took 174s
    # at just 10,000 nodes/50,000 edges, and was still running after 66
    # minutes at 40,000 nodes/200,000 edges. A plain nx.Graph can't have
    # duplicate edges between the same pair, and every node here is added
    # exactly once by the caller, so none of pyvis's duplicate-checking is
    # actually needed.
    for node_id, data in graph.nodes(data=True):
        opts = dict(data)
        opts.setdefault("size", default_node_size)
        opts["size"] = int(opts["size"])
        label = opts.pop("label", None) or node_id
        shape = opts.pop("shape", "dot")
        color = opts.pop("color", "#97c2fc")
        if "group" in opts:
            n = Node(node_id, shape, label=label, font_color=net.font_color, **opts)
        else:
            n = Node(node_id, shape, label=label, color=color, font_color=net.font_color, **opts)
        net.nodes.append(n.options)
        net.node_ids.append(node_id)
        net.node_map[node_id] = n.options

    for source, target, data in graph.edges(data=True):
        opts = dict(data)
        if "weight" in opts and "width" not in opts:
            opts["width"] = opts.pop("weight")
        elif "weight" not in opts and "width" not in opts:
            opts["width"] = default_edge_weight
        e = Edge(source, target, net.directed, **opts)
        net.edges.append(e.options)


def render_network(graph, out_path, notebook=False, inject_filters=None, physics=True):
    net = Network(height="800px", width="100%", notebook=notebook, cdn_resources="in_line")
    net.set_options(_NETWORK_OPTIONS_JSON if physics else _STATIC_NETWORK_OPTIONS_JSON)
    _fast_populate_network(net, graph)
    net.write_html(out_path, notebook=notebook)
    _strip_bootstrap_cdn(out_path)
    inject_filters(out_path, graph)
    if physics:
        _inject_stabilize_then_stop(out_path)
    print(f"  wrote {out_path} ({graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges)")
    return net


# copied from ao3_tag_analysis.py -- keep in sync if that file changes
def build_cluster_graph(pair_stats, keep_tags):
    graph = nx.Graph()
    for tag_id in keep_tags:
        field, _, label = tag_id.partition("::")
        graph.add_node(tag_id, label=label, group=field,
                        color=FIELD_COLORS[field], title=tag_id)
    for row in pair_stats[pair_stats["pmi"] > 0].itertuples(index=False):
        graph.add_edge(
            row.tag_a, row.tag_b, weight=float(row.pmi),
            pmi=float(row.pmi), lift=float(row.lift),
            joint_count=int(row.joint_count),
            title=(f"{row.tag_a} × {row.tag_b}: more likely than chance "
                   f"(lift={row.lift:.2f}, pmi={row.pmi:.2f}, "
                   f"joint count={int(row.joint_count)})"),
        )
    return graph


def detect_communities(graph, resolution=1.0):
    return louvain_communities(graph, weight="weight", resolution=resolution,
                                seed=_CLUSTER_SEED)


def merge_small_communities(communities, graph, min_cluster_size):
    # Tracked via dicts keyed by a stable community id + two lazily-
    # invalidated heaps (by size), not a plain list -- turns every per-merge
    # lookup into O(1)/O(log n) instead of an O(num communities) rescan,
    # which used to make this quadratic in the number of communities
    # (confirmed: 46.9s at 12,005 communities, 139.9s for the
    # fully-isolated-source fallback path alone at 20,001 communities --
    # --all-tags runs producing many small/isolated communities hit exactly
    # this and could stall effectively indefinitely at real scale).
    groups = {i: set(c) for i, c in enumerate(communities)}
    node_owner = {}
    group_min_tag = {}
    for community_id, members in groups.items():
        group_min_tag[community_id] = min(members)
        for node in members:
            node_owner[node] = community_id

    undersized_heap = [(len(members), group_min_tag[community_id], community_id)
                        for community_id, members in groups.items()
                        if len(members) < min_cluster_size]
    heapq.heapify(undersized_heap)
    size_heap = [(-len(members), group_min_tag[community_id], community_id)
                 for community_id, members in groups.items()]
    heapq.heapify(size_heap)

    def current_largest():
        while True:
            neg_size, _, community_id = size_heap[0]
            if community_id in groups and len(groups[community_id]) == -neg_size:
                return community_id
            heapq.heappop(size_heap)

    while len(groups) > 1 and undersized_heap:
        size, _, source_id = heapq.heappop(undersized_heap)
        if source_id not in groups or len(groups[source_id]) != size or size >= min_cluster_size:
            continue  # stale entry -- merged away, or grown since it was pushed

        source = groups.pop(source_id)
        source_min_tag = group_min_tag.pop(source_id)

        weight_by_target = {}
        for node in source:
            for neighbor, edge_data in graph[node].items():
                target_id = node_owner.get(neighbor)
                if target_id is None or target_id == source_id or target_id not in groups:
                    continue
                weight_by_target[target_id] = weight_by_target.get(target_id, 0.0) + edge_data["weight"]

        if weight_by_target:
            best_weight = max(weight_by_target.values())
            candidates = [cid for cid, weight in weight_by_target.items() if weight == best_weight]
            target_id = min(candidates, key=lambda cid: group_min_tag[cid])
        else:
            target_id = current_largest()

        groups[target_id] |= source
        for node in source:
            node_owner[node] = target_id
        group_min_tag[target_id] = min(group_min_tag[target_id], source_min_tag)

        new_size = len(groups[target_id])
        if new_size < min_cluster_size:
            heapq.heappush(undersized_heap, (new_size, group_min_tag[target_id], target_id))
        heapq.heappush(size_heap, (-new_size, group_min_tag[target_id], target_id))

    return list(groups.values())


def assign_cluster_ids(communities):
    ordered = sorted(communities, key=lambda c: (-len(c), min(c) if c else ""))
    rows = []
    for cluster_id, community in enumerate(ordered, start=1):
        for tag_id in community:
            field, _, label = tag_id.partition("::")
            rows.append({"tag_id": tag_id, "field": field, "label": label,
                          "cluster_id": cluster_id})
    result = pd.DataFrame(rows, columns=["tag_id", "field", "label", "cluster_id"])
    return result.sort_values(["cluster_id", "tag_id"]).reset_index(drop=True)


def color_graph_by_cluster(graph, clusters_df):
    cluster_by_tag = clusters_df.set_index("tag_id")["cluster_id"]
    for tag_id, cluster_id in cluster_by_tag.items():
        node = graph.nodes[tag_id]
        node["group"] = str(cluster_id)
        node["color"] = _CLUSTER_PALETTE[(cluster_id - 1) % len(_CLUSTER_PALETTE)]
        node["title"] = f"{tag_id} (cluster {cluster_id})"
    return graph


def _all_cluster_tags(graph):
    return [{"id": node_id, "label": data["label"], "field": node_id.partition("::")[0]}
            for node_id, data in graph.nodes(data=True)]


def _cluster_filter_controls_html(graph):
    all_tags = _all_cluster_tags(graph)
    all_tags_json = json.dumps(all_tags).replace("</script", "<\\/script")

    cluster_ids = sorted({data["group"] for _, data in graph.nodes(data=True)}, key=int)
    checkbox_items = "\n".join(
        '<label class="ao3-cat-label">'
        f'<input type="checkbox" class="ao3-cat-checkbox" data-group="{cluster_id}" checked>'
        f'<span class="ao3-swatch" style="background-color:'
        f'{_CLUSTER_PALETTE[(int(cluster_id) - 1) % len(_CLUSTER_PALETTE)]};"></span>'
        f'Cluster {cluster_id}</label>'
        for cluster_id in cluster_ids
    )
    panel_html = _TAG_PAIR_FILTER_PANEL_HTML.replace("__CHECKBOX_ITEMS__", checkbox_items)
    script_html = _TAG_PAIR_FILTER_SCRIPT_TEMPLATE.replace("__ALL_TAGS_JSON__", all_tags_json)
    return panel_html, script_html


def _inject_cluster_filter_controls(html_path, graph):
    with open(html_path, encoding="utf-8") as f:
        html = f.read()
    assert html.count("<body>") == 1, "expected exactly one <body> tag"
    assert html.count("</body>") == 1, "expected exactly one </body> tag"
    panel_html, script_html = _cluster_filter_controls_html(graph)
    html = html.replace("<body>", "<body>" + panel_html, 1)
    html = html.replace("</body>", script_html + "</body>", 1)
    with open(html_path, "w", encoding="utf-8") as f:
        f.write(html)


def compute_cluster_layout(graph, clusters_df, node_spacing=40):
    # Static (x, y) position for every node instead of relying on
    # vis-network's client-side physics simulation -- confirmed directly
    # that plain nx.spring_layout on the WHOLE graph took 72s at just
    # 5,000 nodes, and a global force simulation running in the browser is
    # exactly what makes a real tens-of-thousands-of-nodes network hang or
    # crash the tab, independent of file size.
    #
    # Arranges each cluster's nodes on its own circle (nx.circular_layout,
    # radius scaled by sqrt(cluster size) so point density stays roughly
    # comparable across differently-sized clusters) and places the circles
    # on a grid, sized so no two clusters can overlap. Deliberately NOT
    # spring_layout, even per-cluster: its cost grows worse than linearly
    # (500 nodes: 2.4s, 4000 nodes: 44.3s) and a single oversized cluster
    # -- entirely possible; Louvain can collapse a noisy/weakly-structured
    # graph into a handful of large communities -- would make the whole
    # layout step's cost unbounded again. circular_layout is a closed-form
    # O(1)-per-node placement with no iteration, confirmed instant even
    # for 40,000 nodes in a single cluster.
    # Shelf packing, NOT a uniform grid sized by the largest cluster: real
    # Louvain output at --all-tags scale is one or a few giant communities
    # plus thousands of tiny (3-6 tag) ones, and a uniform grid spread that
    # across an ~800,000-px canvas -- vis-network's fit() then zooms out so
    # far every node paints at ~0.02px and the page looks completely blank.
    # Shelf packing keeps the span near sqrt(total cluster area).
    cluster_groups = clusters_df.groupby("cluster_id")["tag_id"].apply(list)

    entries = []
    for cluster_id, tag_ids in cluster_groups.items():
        radius_units = max(1.0, math.sqrt(len(tag_ids)))
        diameter_px = (2 * radius_units + 2) * node_spacing
        entries.append((diameter_px, cluster_id, tag_ids, radius_units))
    if not entries:
        return {}

    entries.sort(key=lambda e: (-e[0], e[1]))
    total_area = sum(diameter * diameter for diameter, _, _, _ in entries)
    target_width = max(entries[0][0], math.sqrt(total_area))

    positions = {}
    shelf_x = 0.0
    shelf_y = 0.0
    shelf_height = 0.0
    for diameter, cluster_id, tag_ids, radius_units in entries:
        if shelf_x > 0 and shelf_x + diameter > target_width:
            shelf_y += shelf_height
            shelf_x = 0.0
            shelf_height = 0.0
        center_x = shelf_x + diameter / 2
        center_y = shelf_y + diameter / 2
        shelf_x += diameter
        shelf_height = max(shelf_height, diameter)

        subgraph = graph.subgraph(tag_ids)
        if len(subgraph) == 1:
            local_pos = {next(iter(subgraph)): (0.0, 0.0)}
        else:
            local_pos = nx.circular_layout(subgraph, scale=radius_units)
        for tag_id, (x, y) in local_pos.items():
            positions[tag_id] = (x * node_spacing + center_x, y * node_spacing + center_y)
    return positions


cluster_graph = build_cluster_graph(pair_stats, keep_tags)
communities = detect_communities(cluster_graph, resolution=CLUSTER_RESOLUTION)
communities = merge_small_communities(communities, cluster_graph, MIN_CLUSTER_SIZE)
clusters_df = assign_cluster_ids(communities)
clusters_df.to_csv(CLUSTERS_OUT, index=False)
print(f"wrote {CLUSTERS_OUT} ({len(clusters_df)} tags, "
      f"{clusters_df['cluster_id'].nunique()} clusters)")

iframe = None
if not clusters_df.empty:
    color_graph_by_cluster(cluster_graph, clusters_df)
    positions = compute_cluster_layout(cluster_graph, clusters_df)
    for tag_id, (x, y) in positions.items():
        cluster_graph.nodes[tag_id]["x"] = x
        cluster_graph.nodes[tag_id]["y"] = y
    net = render_network(cluster_graph, CLUSTER_NETWORK_OUT, notebook=True,
                          inject_filters=_inject_cluster_filter_controls, physics=False)
    iframe = net.show(CLUSTER_NETWORK_OUT, notebook=True)
    # net.show() internally rewrites the file from pyvis's raw template,
    # wiping out the post-processing above -- redo it on the file it just
    # wrote, same as ao3_tag_visualizer.py's tag-pair network cell.
    _strip_bootstrap_cdn(CLUSTER_NETWORK_OUT)
    _inject_cluster_filter_controls(CLUSTER_NETWORK_OUT, cluster_graph)

iframe if iframe is not None else clusters_df.head(20)

## Cluster meta-network + Gephi export

The full tag-level network above is faithful but cognitively unreadable at
`ALL_TAGS` scale (tens of thousands of nodes). The meta-network is the
readable summary view: one node per cluster, sized by tag count, labeled with
the cluster's top fandom (from the same per-cluster fandom summary
`ao3_tag_fandom_labels.py` writes), edges weighted by how many positive-PMI
tag pairs cross each cluster pair. Optionally (set `GEXF_OUT`), the full
tag-level graph is also written as GEXF for [Gephi](https://gephi.org/) --
open it there, then Appearance → Partition by `cluster_id` and Layout →
ForceAtlas2.

In [ ]:
# copied from ao3_tag_analysis.py -- keep in sync if that file changes
def compute_cluster_fandom_summary(df, clusters_df, top_n):
    """Per-cluster fandom summary (lives here rather than in
    ao3_tag_fandom_labels.py -- which re-exports it -- because the cluster
    meta-network below needs the same labels, and that module imports this
    one, so the import can't go the other way): for each cluster_id, pools
    every work containing ANY of the cluster's tags -- counted once per
    cluster no matter how many of its tags the work matches -- and ranks
    the fandoms of those works by percent of works (tie-break:
    alphabetically smallest fandom name). Same denominator semantics as
    ao3_tag_fandom_labels.py's per-tag labels: the percentage base is the
    cluster's full work pool, so fandom-less works keep sums under 100 and
    multi-fandom crossover works can push sums above 100, while every
    individual value stays <= 100. Returns a DataFrame
    [cluster_id, n_tags, n_works, n_fandoms, top_fandoms] with one row per
    cluster in clusters_df -- a cluster whose tags never appear in df keeps
    its row with n_works=0, n_fandoms=0, and an empty label rather than
    vanishing. n_fandoms is how many DISTINCT fandoms the cluster's works
    span (every fandom that appears at least once, not truncated to
    top_n)."""
    cluster_by_tag = clusters_df.drop_duplicates("tag_id").set_index("tag_id")["cluster_id"]

    tag_table = build_document_tag_table(df, fields=ALL_METADATA_FIELDS)
    tag_table = tag_table[tag_table["tag_id"].isin(cluster_by_tag.index)]
    cluster_works = tag_table.assign(cluster_id=tag_table["tag_id"].map(cluster_by_tag))
    # A work with several of the cluster's tags is still one story.
    cluster_works = cluster_works[["cluster_id", "work_id"]].drop_duplicates()
    cluster_totals = cluster_works.groupby("cluster_id").size()

    # Deduped for the same reason as compute_fandom_labels: the scraper
    # emits one row per (seed tag, work).
    deduped = df.drop_duplicates(subset="work_id", keep="first")
    fandom_table = explode_field(deduped, "fandom")[["work_id", "fandom"]]

    merged = cluster_works.merge(fandom_table, on="work_id")
    counts = merged.groupby(["cluster_id", "fandom"]).size().reset_index(name="count")
    counts["pct"] = counts["count"] / counts["cluster_id"].map(cluster_totals) * 100

    cluster_fandoms = counts.groupby("cluster_id")["fandom"].nunique()

    counts = counts.sort_values(["cluster_id", "count", "fandom"], ascending=[True, False, True])
    top = counts.groupby("cluster_id", sort=False).head(top_n)
    top = top.assign(entry=top["fandom"] + " (" + top["pct"].round(0).astype(int).astype(str) + "%)")
    labels = top.groupby("cluster_id", sort=False)["entry"].apply(", ".join)

    n_tags = clusters_df.groupby("cluster_id")["tag_id"].nunique()
    summary = pd.DataFrame({
        "cluster_id": n_tags.index,
        "n_tags": n_tags.values,
        "n_works": n_tags.index.map(cluster_totals).fillna(0).astype(int),
        "n_fandoms": n_tags.index.map(cluster_fandoms).fillna(0).astype(int),
        "top_fandoms": n_tags.index.map(labels).fillna(""),
    })
    # ao3_tag_clusters.csv's cluster_ids are integers, but they arrive as
    # strings when the CSV is read with dtype=str -- order numerically when
    # every id parses as a number, so 2 sorts before 10.
    numeric_order = pd.to_numeric(summary["cluster_id"], errors="coerce")
    if numeric_order.notna().all():
        summary = summary.iloc[numeric_order.argsort(kind="stable").to_numpy()]
    else:
        summary = summary.sort_values("cluster_id")
    return summary.reset_index(drop=True)


def build_cluster_meta_graph(pair_stats, clusters_df, fandom_summary):
    summary_by_id = fandom_summary.set_index("cluster_id")

    graph = nx.Graph()
    for cluster_id, row in summary_by_id.iterrows():
        top_fandom = row["top_fandoms"].split(" (")[0] if row["top_fandoms"] else ""
        label = f"{cluster_id}: {top_fandom}" if top_fandom else f"Cluster {cluster_id}"
        graph.add_node(
            f"cluster::{cluster_id}", label=label, group=str(cluster_id),
            color=_CLUSTER_PALETTE[(int(cluster_id) - 1) % len(_CLUSTER_PALETTE)],
            size=int(min(60, 10 + 2 * math.sqrt(row["n_tags"]))),
            title=(f"Cluster {cluster_id} — {row['n_tags']} tags, "
                   f"{row['n_works']} works — {row['top_fandoms'] or 'no fandom co-occurrence'}"),
        )

    cluster_by_tag = clusters_df.drop_duplicates("tag_id").set_index("tag_id")["cluster_id"]
    positive = pair_stats[pair_stats["pmi"] > 0]
    cluster_a = positive["tag_a"].map(cluster_by_tag)
    cluster_b = positive["tag_b"].map(cluster_by_tag)
    crossing = positive.assign(cluster_a=cluster_a, cluster_b=cluster_b)
    crossing = crossing[crossing["cluster_a"].notna() & crossing["cluster_b"].notna()
                         & (crossing["cluster_a"] != crossing["cluster_b"])]
    # Canonicalize so (3, 7) and (7, 3) aggregate together.
    lo = crossing[["cluster_a", "cluster_b"]].min(axis=1)
    hi = crossing[["cluster_a", "cluster_b"]].max(axis=1)
    crossing = crossing.assign(cluster_lo=lo, cluster_hi=hi)
    grouped = crossing.groupby(["cluster_lo", "cluster_hi"]).agg(
        n_links=("pmi", "size"), mean_pmi=("pmi", "mean"))

    for (cluster_lo, cluster_hi), row in grouped.iterrows():
        graph.add_edge(
            f"cluster::{cluster_lo}", f"cluster::{cluster_hi}",
            n_links=int(row["n_links"]), mean_pmi=float(row["mean_pmi"]),
            width=min(10.0, 1 + math.log1p(row["n_links"])),
            title=(f"Cluster {cluster_lo} × Cluster {cluster_hi}: "
                   f"{int(row['n_links'])} positive-PMI tag pairs, "
                   f"mean pmi={row['mean_pmi']:.2f}"),
        )
    return graph


def write_gexf_export(graph, clusters_df, out_path):
    cluster_by_tag = clusters_df.drop_duplicates("tag_id").set_index("tag_id")["cluster_id"]

    export = nx.Graph()
    for tag_id in graph.nodes():
        field, _, label = tag_id.partition("::")
        export.add_node(tag_id, label=label, field=field,
                         cluster_id=int(cluster_by_tag.get(tag_id, 0)))
    for tag_a, tag_b, data in graph.edges(data=True):
        export.add_edge(tag_a, tag_b, weight=float(data["pmi"]),
                         lift=float(data["lift"]), joint_count=int(data["joint_count"]))
    nx.write_gexf(export, out_path)
    print(f"wrote {out_path} ({export.number_of_nodes()} nodes, "
          f"{export.number_of_edges()} edges)")


meta_iframe = None
if not clusters_df.empty:
    fandom_summary = compute_cluster_fandom_summary(df, clusters_df, top_n=3)
    meta_graph = build_cluster_meta_graph(pair_stats, clusters_df, fandom_summary)
    meta_net = render_network(meta_graph, CLUSTER_META_NETWORK_OUT, notebook=True,
                               inject_filters=_inject_cluster_filter_controls)
    meta_iframe = meta_net.show(CLUSTER_META_NETWORK_OUT, notebook=True)
    # net.show() internally rewrites the file from pyvis's raw template,
    # wiping out the post-processing above -- redo it on the file it just
    # wrote, same as the tag-level network cell.
    _strip_bootstrap_cdn(CLUSTER_META_NETWORK_OUT)
    _inject_cluster_filter_controls(CLUSTER_META_NETWORK_OUT, meta_graph)
    _inject_stabilize_then_stop(CLUSTER_META_NETWORK_OUT)

    if GEXF_OUT:
        write_gexf_export(cluster_graph, clusters_df, GEXF_OUT)

meta_iframe

## Done

`{FREQUENCY_OUT}`, `{CLUSTER_NETWORK_OUT}`, `{CLUSTER_META_NETWORK_OUT}`, and
`{CLUSTERS_OUT}` are now in the notebook's working directory (plus
`{GEXF_OUT}` for Gephi, if set). Re-run from the Configuration cell with
different `FREQUENCY_TOP_N`/`FREQUENCY_BOTTOM_N`/`FREQUENCY_MIN_COUNT`/
`TOP_TAGS`/`MIN_PAIR_COUNT`/`CLUSTER_RESOLUTION`/`MIN_CLUSTER_SIZE` values to
adjust what's included.